<a href="https://colab.research.google.com/github/sanjil18/Fine-tuning-project/blob/main/03_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 3 — Evaluation: Base Model vs. Fine-Tuned Model

Runs a fixed set of evaluation questions (in-training-data, out-of-training-data, and general-knowledge questions) through:
1. The **base** Qwen2.5-1.5B-Instruct model (no adapter)
2. The **fine-tuned** model (base + your 150-example LoRA adapter)

Records each response and its latency, then exports `results/model_outputs.csv` for local comparison against your RAG system.

**Before running:** `Runtime > Change runtime type > T4 GPU`.

**You'll need to upload two things when prompted:**
- `eval_questions.jsonl`
- `lora_adapter_150ex.zip` (zip your `models/lora_adapter_150ex/` folder before uploading)

## 1. Install dependencies

In [1]:
!pip install -q -U transformers accelerate peft bitsandbytes pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 63.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 53.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.6 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.6 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.6 which is incompatible.


## 2. Upload eval questions and the LoRA adapter zip

In [2]:
from google.colab import files

print("Upload eval_questions.jsonl:")
uploaded_eval = files.upload()
EVAL_PATH = list(uploaded_eval.keys())[0]

print("\nUpload lora_adapter_150ex.zip:")
uploaded_adapter = files.upload()
ADAPTER_ZIP = list(uploaded_adapter.keys())[0]

import zipfile, os
ADAPTER_DIR = "lora_adapter_150ex"
os.makedirs(ADAPTER_DIR, exist_ok=True)
with zipfile.ZipFile(ADAPTER_ZIP, "r") as z:
    z.extractall(ADAPTER_DIR)

print(f"\nEval questions: {EVAL_PATH}")
print(f"Adapter extracted to: {ADAPTER_DIR}")
print(os.listdir(ADAPTER_DIR))

Upload eval_questions.jsonl:


Saving eval_questions.jsonl to eval_questions.jsonl

Upload lora_adapter_150ex.zip:


Saving lora_adapter_final (1).zip to lora_adapter_final (1).zip

Eval questions: eval_questions.jsonl
Adapter extracted to: lora_adapter_150ex
['README.md', 'chat_template.jinja', 'tokenizer_config.json', 'training_args.bin', 'tokenizer.json', 'adapter_model.safetensors', 'adapter_config.json']


## 3. Config

In [3]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
MAX_NEW_TOKENS = 200   # higher than the Week 2 sanity check so answers don't get cut off
OUTPUT_CSV = "model_outputs.csv"

## 4. Load eval questions

In [4]:
import json

questions = []
with open(EVAL_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            questions.append(json.loads(line))

print(f"Loaded {len(questions)} eval questions")
for q in questions:
    print(f"  [{q['category']}] {q['question']}")

Loaded 18 eval questions
  [in_training] What are ACID properties in relational databases?
  [in_training] Why are B-tree indexes preferred over hash indexes for range queries?
  [in_training] What is the vanishing gradient problem and which architecture was designed to address it?
  [in_training] What is the difference between a Python decorator and a context manager?
  [in_training] What does self-attention compute for each word in a transformer?
  [in_training] Why does overfitting happen and how can Early Stopping help prevent it?
  [out_of_training] What is database sharding and why is it used?
  [out_of_training] How do Generative Adversarial Networks (GANs) work at a high level?
  [out_of_training] What is the difference between Python's async/await and traditional multithreading?
  [out_of_training] What is sparse attention and how does it differ from standard self-attention?
  [out_of_training] What is database replication and how does it differ from sharding?
  [out_of_traini

## 5. Load base model (4-bit)

In [5]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)
base_model.eval()
print("Base model loaded.")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Base model loaded.


## 6. Helper function to ask a question and time it

In [6]:
import time

def ask(model, question, max_new_tokens=MAX_NEW_TOKENS):
    messages = [{"role": "user", "content": question}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    start = time.time()
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    latency = time.time() - start

    response = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return response.strip(), round(latency, 2)

## 7. Run all questions through the BASE model

In [7]:
results = []

for i, q in enumerate(questions, 1):
    print(f"[{i}/{len(questions)}] BASE — {q['question']}")
    answer, latency = ask(base_model, q["question"])
    results.append({
        "question": q["question"],
        "category": q["category"],
        "base_model_answer": answer,
        "base_model_latency_sec": latency,
    })
    print(f"  -> {latency}s\n")

print("Done with base model.")

[1/18] BASE — What are ACID properties in relational databases?
  -> 12.63s

[2/18] BASE — Why are B-tree indexes preferred over hash indexes for range queries?
  -> 11.45s

[3/18] BASE — What is the vanishing gradient problem and which architecture was designed to address it?
  -> 11.44s

[4/18] BASE — What is the difference between a Python decorator and a context manager?
  -> 9.01s

[5/18] BASE — What does self-attention compute for each word in a transformer?
  -> 18.24s

[6/18] BASE — Why does overfitting happen and how can Early Stopping help prevent it?
  -> 11.48s

[7/18] BASE — What is database sharding and why is it used?
  -> 10.42s

[8/18] BASE — How do Generative Adversarial Networks (GANs) work at a high level?
  -> 11.4s

[9/18] BASE — What is the difference between Python's async/await and traditional multithreading?
  -> 11.3s

[10/18] BASE — What is sparse attention and how does it differ from standard self-attention?
  -> 11.38s

[11/18] BASE — What is database repl

## 8. Load the LoRA adapter on top of the base model

In [8]:
from peft import PeftModel

finetuned_model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
finetuned_model.eval()
print("Fine-tuned model (base + LoRA adapter) loaded.")

Fine-tuned model (base + LoRA adapter) loaded.


## 9. Run all questions through the FINE-TUNED model

In [9]:
for i, r in enumerate(results, 1):
    print(f"[{i}/{len(results)}] FINETUNED — {r['question']}")
    answer, latency = ask(finetuned_model, r["question"])
    r["finetuned_answer"] = answer
    r["finetuned_latency_sec"] = latency
    print(f"  -> {latency}s\n")

print("Done with fine-tuned model.")

[1/18] FINETUNED — What are ACID properties in relational databases?
  -> 9.23s

[2/18] FINETUNED — Why are B-tree indexes preferred over hash indexes for range queries?
  -> 8.69s

[3/18] FINETUNED — What is the vanishing gradient problem and which architecture was designed to address it?
  -> 5.46s

[4/18] FINETUNED — What is the difference between a Python decorator and a context manager?
  -> 7.46s

[5/18] FINETUNED — What does self-attention compute for each word in a transformer?
  -> 4.34s

[6/18] FINETUNED — Why does overfitting happen and how can Early Stopping help prevent it?
  -> 8.67s

[7/18] FINETUNED — What is database sharding and why is it used?
  -> 6.55s

[8/18] FINETUNED — How do Generative Adversarial Networks (GANs) work at a high level?
  -> 6.8s

[9/18] FINETUNED — What is the difference between Python's async/await and traditional multithreading?
  -> 8.0s

[10/18] FINETUNED — What is sparse attention and how does it differ from standard self-attention?
  -> 4.

## 10. Save results to CSV and download

In [10]:
import pandas as pd

df = pd.DataFrame(results, columns=[
    "question", "category",
    "base_model_answer", "base_model_latency_sec",
    "finetuned_answer", "finetuned_latency_sec",
])
df.to_csv(OUTPUT_CSV, index=False)
print(df[["category", "base_model_latency_sec", "finetuned_latency_sec"]].groupby("category").mean())

files.download(OUTPUT_CSV)

                   base_model_latency_sec  finetuned_latency_sec
category                                                        
general_knowledge                1.898333               1.583333
in_training                     12.375000               7.308333
out_of_training                 10.733333               5.923333


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>